# Final Model Training

Train one selected model on the full Cross training dataset, save the checkpoint, and evaluate it on the held-out Cross test folders.

# Imports

In [17]:
import json
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from torch import nn

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.loader import LABEL_MAP, MEGDataset
from src.test import test_model_on_folders
from src.train import (
    build_model,
    count_parameters,
    format_seconds,
    get_default_device,
    make_loader,
    run_epoch,
    synchronize_device,
)
from src.utilities import list_npy_files

# Config

In [21]:
train_folder = PROJECT_ROOT / "cache" / "base" / "Cross" / "train"
test_folders = [
    PROJECT_ROOT / "cache" / "base" / "Cross" / "test1",
    PROJECT_ROOT / "cache" / "base" / "Cross" / "test2",
    PROJECT_ROOT / "cache" / "base" / "Cross" / "test3",
]

model_dir = PROJECT_ROOT / "results" / "small_cnn"
test_output_csv = model_dir / "test_results.csv"

model_name = "small_cnn"
model_params = {
            "kernel_size": [7],
            "channels": [(8, 16, 32)],
            "dropout": [0.5],
}

training_params = {
       "learning_rate": 3e-4,
    "weight_decay": 1e-3,
    "window_seconds": 7,
    "overlap": 0.5,
    "epochs": 80,
    "patience": 10,
    "batch_size": 128,
    "num_workers": 0,
    "seed": 42,
}

device = get_default_device()
device

device(type='mps')

# Data And Model

In [22]:
seed = training_params.get("seed")
if seed is not None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

train_files = list_npy_files(train_folder)
if not train_files:
    raise FileNotFoundError(f"No .npy files found in {train_folder}. Run notebooks/data.ipynb first.")

train_dataset = MEGDataset(
    train_files,
    window_seconds=training_params["window_seconds"],
    overlap=training_params.get("overlap", 0.5),
)
train_loader = make_loader(
    train_dataset,
    batch_size=training_params["batch_size"],
    shuffle=True,
    num_workers=training_params["num_workers"],
    device=device,
)

sample_x, _ = train_dataset[0]
num_classes = len(LABEL_MAP)
model = build_model(
    model_name=model_name,
    input_channels=sample_x.shape[0],
    input_time=sample_x.shape[1],
    num_classes=num_classes,
    model_params=model_params,
).to(device)

print(f"Device: {device}")
print(f"Files: {len(train_files)}")
print(f"Windows: {len(train_dataset)}")
print(f"Sample shape: {tuple(sample_x.shape)}")
print(f"Parameters: {count_parameters(model):,}")

ValueError: Unknown model: small_cnn

# Train

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=training_params["learning_rate"],
    weight_decay=training_params["weight_decay"],
)
use_amp = training_params.get("use_amp", True) and device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

history = []
best_train_loss = float("inf")
best_epoch = None
best_model_state_dict = None
start = time.perf_counter()

for epoch in range(1, training_params["epochs"] + 1):
    synchronize_device(device)
    epoch_start = time.perf_counter()

    train_loss = run_epoch(model, train_loader, criterion, optimizer, device, scaler, use_amp)

    synchronize_device(device)
    epoch_seconds = time.perf_counter() - epoch_start
    history.append({"epoch": epoch, "train_loss": train_loss, "epoch_seconds": epoch_seconds})

    if train_loss < best_train_loss:
        best_train_loss = train_loss
        best_epoch = epoch
        best_model_state_dict = {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.state_dict().items()
        }

    print(
        f"epoch {epoch}/{training_params['epochs']} | "
        f"train_loss={train_loss:.4f} | "
        f"time={format_seconds(epoch_seconds)}"
    )

total_seconds = time.perf_counter() - start
model.load_state_dict(best_model_state_dict)
print(f"Finished final training in {format_seconds(total_seconds)}")
print(f"Best epoch by train_loss: {best_epoch} | train_loss={best_train_loss:.4f}")

epoch 1/150 | train_loss=0.1496 | time=0.1s
epoch 2/150 | train_loss=0.0512 | time=0.1s
epoch 3/150 | train_loss=0.0888 | time=0.1s
epoch 4/150 | train_loss=0.0511 | time=0.1s
epoch 5/150 | train_loss=0.0272 | time=0.1s
epoch 6/150 | train_loss=0.0423 | time=0.1s
epoch 7/150 | train_loss=0.0272 | time=0.1s
epoch 8/150 | train_loss=0.0169 | time=0.1s
epoch 9/150 | train_loss=0.0192 | time=0.1s
epoch 10/150 | train_loss=0.0199 | time=0.1s
epoch 11/150 | train_loss=0.0118 | time=0.1s
epoch 12/150 | train_loss=0.0084 | time=0.1s
epoch 13/150 | train_loss=0.0106 | time=0.1s
epoch 14/150 | train_loss=0.0068 | time=0.1s
epoch 15/150 | train_loss=0.0105 | time=0.1s
epoch 16/150 | train_loss=0.0105 | time=0.1s
epoch 17/150 | train_loss=0.0048 | time=0.1s
epoch 18/150 | train_loss=0.0076 | time=0.1s
epoch 19/150 | train_loss=0.0056 | time=0.1s
epoch 20/150 | train_loss=0.0043 | time=0.1s
epoch 21/150 | train_loss=0.0032 | time=0.1s
epoch 22/150 | train_loss=0.0030 | time=0.1s
epoch 23/150 | trai

# Save Model

In [14]:
def to_jsonable(value):
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, dict):
        return {key: to_jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    return value

model_dir.mkdir(parents=True, exist_ok=True)
history_df = pd.DataFrame(history)
history_df.to_csv(model_dir / "training_history.csv", index=False)
model.load_state_dict(best_model_state_dict)

metadata = {
    "model_name": model_name,
    "model_params": to_jsonable(model_params),
    "training_params": to_jsonable(training_params),
    "label_map": LABEL_MAP,
    "input_channels": sample_x.shape[0],
    "input_time": sample_x.shape[1],
    "num_classes": num_classes,
    "number_of_parameters": count_parameters(model),
    "checkpoint_selection": "lowest_train_loss",
    "best_epoch": best_epoch,
    "best_train_loss": best_train_loss,
    "final_train_loss": history[-1]["train_loss"],
    "total_seconds": total_seconds,
}

with open(model_dir / "config.json", "w") as f:
    json.dump(metadata, f, indent=2)

torch.save(
    {
        "model_state_dict": best_model_state_dict,
        "optimizer_state_dict": optimizer.state_dict(),
        "metadata": metadata,
    },
    model_dir / "model.pt",
)

print(f"Saved final model to {model_dir.resolve()}")

Saved final model to /Users/fabiodijkshoorn/Documents/UU/Deep Learning/assignment-2-final/trained_models/final2_cross_cnn


# Test Saved Model

In [15]:
test_results = test_model_on_folders(
    checkpoint_path=model_dir / "model.pt",
    test_folders=test_folders,
    output_csv=test_output_csv,
    batch_size=training_params["batch_size"],
    num_workers=training_params["num_workers"],
    device=device,
)

test_results

Evaluated /Users/fabiodijkshoorn/Documents/UU/Deep Learning/assignment-2-final/cache/Cross/test1 | windows=64 | accuracy=0.4531 | macro_f1=0.4802 | time=0.1s
Evaluated /Users/fabiodijkshoorn/Documents/UU/Deep Learning/assignment-2-final/cache/Cross/test2 | windows=64 | accuracy=0.4844 | macro_f1=0.4762 | time=0.0s
Evaluated /Users/fabiodijkshoorn/Documents/UU/Deep Learning/assignment-2-final/cache/Cross/test3 | windows=64 | accuracy=0.7812 | macro_f1=0.7665 | time=0.0s
Evaluated combined test folders | windows=192 | accuracy=0.5729 | macro_f1=0.5780 | time=0.1s
Saved test results to /Users/fabiodijkshoorn/Documents/UU/Deep Learning/assignment-2-final/trained_models/final2_cross_cnn/test_results.csv
Finished testing in 0.2s


,split,folder,num_files,num_windows,model,device,number_of_parameters,seconds,kernel_size,channels,...,rest_f1,task_story_math_precision,task_story_math_recall,task_story_math_f1,task_working_memory_precision,task_working_memory_recall,task_working_memory_f1,task_motor_precision,task_motor_recall,task_motor_f1
0,test1,/Users/fabiodijkshoorn/Documents/UU/Deep Learn...,16,64,cnn,mps,128420,0.069910,7,"[32, 64, 128]",...,0.814815,0.500000,0.437500,0.466667,0.571429,0.250000,0.347826,0.218750,0.437500,0.291667
1,test2,/Users/fabiodijkshoorn/Documents/UU/Deep Learn...,16,64,cnn,mps,128420,0.039408,7,"[32, 64, 128]",...,0.933333,1.000000,0.125000,0.222222,0.318182,0.437500,0.368421,0.307692,0.500000,0.380952
2,test3,/Users/fabiodijkshoorn/Documents/UU/Deep Learn...,16,64,cnn,mps,128420,0.034436,7,"[32, 64, 128]",...,0.969697,1.000000,0.437500,0.608696,0.687500,0.687500,0.687500,0.666667,1.000000,0.800000
3,combined,"[""/Users/fabiodijkshoorn/Documents/UU/Deep Lea...",48,192,cnn,mps,128420,0.050343,7,"[32, 64, 128]",...,0.911111,0.695652,0.333333,0.450704,0.488889,0.458333,0.473118,0.378049,0.645833,0.476923


# Quick Check

In [16]:
model.eval()
x, y = next(iter(train_loader))
x = x.to(device)

with torch.no_grad():
    logits = model(x)

print(f"batch: {tuple(x.shape)}")
print(f"logits: {tuple(logits.shape)}")
print(f"checkpoint: {model_dir / 'model.pt'}")
print(f"test csv: {test_output_csv}")

batch: (128, 248, 279)
logits: (128, 4)
checkpoint: /Users/fabiodijkshoorn/Documents/UU/Deep Learning/assignment-2-final/trained_models/final2_cross_cnn/model.pt
test csv: /Users/fabiodijkshoorn/Documents/UU/Deep Learning/assignment-2-final/trained_models/final2_cross_cnn/test_results.csv
